# Friends Demo — Phase 4: Decay & Reinforcement

Phase 3 turned decay off on purpose. This notebook turns it on and tests mem0's real,
documented decay feature directly:

- Soft re-rank applied only at search time -- storage is untouched
- Each memory tracks up to its last 20 access timestamps
- Recently-accessed memories get up to a `1.5x` boost; idle ones dampen toward `0.3x`
- Nothing is ever fully hidden -- floor is `0.3x`, not `0x`

Same Maya/Jordan/Sam data, continued from Phase 3. We already have a stale fact (pottery) and
a fresh one (painting) sitting there from Phase 2 -- no new setup needed to start.


## 0. Baseline, decay off

In [ ]:
import os
from dotenv import load_dotenv
from mem0 import MemoryClient

load_dotenv()
client = MemoryClient(api_key=os.getenv("MEM0_API_KEY"))
client.project.update(decay=False)

query = "what hobby is Maya doing these days?"
baseline = client.search(query=query, filters={"user_id": "maya"}, top_k=10)

print("BASELINE (decay off):")
for r in baseline.get("results", []):
    print(f"{r['score']:.3f}  {r['memory']}")

## 1. Turn decay on

In [ ]:
client.project.update(decay=True)
print("Decay enabled for this project.")

## 2. Immediate re-run -- the fallback case

Every fact so far predates decay being turned on, so this first call after flipping the
toggle uses a one-time fallback (last-update timestamp) rather than real access history.
Expect a small or no change from baseline here.


In [ ]:
immediately_after = client.search(query=query, filters={"user_id": "maya"}, top_k=10)

print("IMMEDIATELY AFTER enabling decay:")
for r in immediately_after.get("results", []):
    print(f"{r['score']:.3f}  {r['memory']}")

## 3. Reinforcement experiment

Decay rewards *access*, not just age. We repeatedly search in a way that surfaces the
painting fact, and leave the pottery fact untouched, then compare.


In [ ]:
for _ in range(5):
    client.search(query="Maya's painting studio on Thursdays", filters={"user_id": "maya"}, top_k=1)

print("Reinforced the painting fact 5 times via repeated search.")

In [ ]:
reinforced = client.search(query=query, filters={"user_id": "maya"}, top_k=10)

print("AFTER reinforcing the painting fact:")
for r in reinforced.get("results", []):
    print(f"{r['score']:.3f}  {r['memory']}")

print("\nIf reinforcement is working, painting's score should have moved up relative to")
print("pottery, compared to Section 2's numbers. Don't assume it did -- check the numbers.")

## 4. Threshold + decay interaction

Threshold filtering happens **before** decay's scaling factor is applied. A stale-but-relevant
memory that cleared the threshold can still show up with a *final* score below it -- it stays
visible, just visibly dampened.


In [ ]:
thresholded = client.search(query=query, filters={"user_id": "maya"}, threshold=0.5, top_k=10)

print("threshold=0.5, decay on:")
for r in thresholded.get("results", []):
    below = "  <-- below 0.5 despite the threshold!" if r["score"] < 0.5 else ""
    print(f"{r['score']:.3f}  {r['memory']}{below}")

If several results print below 0.5, that's not a bug -- it means the threshold check
happened against the raw relevance score before decay dampened it for display. See Phase 3's
Section 4 for the same mechanism tested with pure relevance, no decay.


## 5. What decay does *not* do

In [ ]:
print("Floor scaling factor: 0.3x (never fully hidden)")
print("Ceiling scaling factor: 1.5x")
print("Affects: search-time ranking only")
print("Does NOT affect: add(), storage, or embeddings")

## Discussion — is there really a "right half-life" for a fact?

This is worth treating as an open question, not a solved mechanism. Whatever Section 3 above
actually showed is the real evidence: if painting's score didn't clearly overtake pottery's
even after 5x reinforcement, that means a single fixed decay curve doesn't obviously know the
right half-life for *this* fact in *this* context. A hobby switch is arguably a fact that
should drop to near-irrelevance almost immediately once contradicted -- not just get a soft
down-weight competing on equal footing with a related-but-stale fact. Worth saying exactly
that in the demo, rather than presenting decay as a settled answer.
